<a href="https://colab.research.google.com/github/schrodinger2/theFrauds/blob/main/ML/preprocessing/comiset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

# Install aria2c
!apt-get -y install aria2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1 [73.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libaria2-0 amd64 1.37.0+debian-1build3 [1,105 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 aria2 amd64 1.37.0+debian-1build3 [387 kB]
Fetched 1,566 kB in 1s (1,371 kB/s)
Selecting previously unselected package libcares2:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubu

In [ ]:
!pip install kaggle

In [ ]:
!aria2c -x 8 -s 8 -o Comiset23_Lab_Environment_Dataset.zip "https://zenodo.org/records/15375146/files/Comiset23_Lab_Environment_Dataset.zip?download=1"


09/22 22:52:43 [NOTICE] Downloading 1 item(s)

09/22 22:53:37 [NOTICE] Download complete: /content/Comiset23_Lab_Environment_Dataset.zip

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
cc57c1|OK  |    88MiB/s|/content/Comiset23_Lab_Environment_Dataset.zip

Status Legend:
(OK):download completed.


In [ ]:
import zipfile
import json
import os

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

# See what's inside the ZIP
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Files in ZIP:")
    for info in z.infolist():
        print(f"{info.filename} | compressed: {info.compress_size/1e9:.2f} GB | "
              f"uncompressed: {info.file_size/1e9:.2f} GB")

    # Pick the JSON file
    json_files = [x for x in z.namelist() if x.lower().endswith(".json")]
    print("\nJSON files:", json_files)

    json_name = json_files[0]

    # Read only the beginning — DO NOT extract the file
    with z.open(json_name) as f:
        for i in range(30):
            line = f.readline()
            if not line:
                break
            print(line[:1000].decode("utf-8", errors="replace"))

Files in ZIP:
dataset_comillas2.json | compressed: 4.91 GB | uncompressed: 159.73 GB

JSON files: ['dataset_comillas2.json']
{"_index":"logs-endpoint-winevent-additional-2022.11.16","_type":"_doc","_id":"8c0acd213cc3524fc717afbc33644d9c2eb6626c","_score":1,"_source":{"event_original_time":"2022-11-16T08:25:38.047Z","etl_processed_time":"2022-11-16T08:28:16.386Z","user_name":"system","PossibleCause":"Unknown","type":"wineventlog","process_id":888,"@timestamp":"2022-11-16T08:25:38.047Z","host_name":"desktop-4pvps6e.phoenix.local","level":"error","etl_host_agent_type":"winlogbeat","etl_pipeline":["all-filter-0098","all-add_processed_timestamp","fingerprint-winlogbeats7","winlogbeat_7_and_above-field_nest_cleanup","winlogbeat_7_and_above-field_cleanups","1500","1522","wmi-user_field_renames","wmi-all-extract_domain_and_user_name","general_rename-various_global_options","provider_guid-cleanup","winevent-hostname-cleanup","winevent-user_name-is-machine-account","final-cleanup-message_field"]

In [ ]:
import zipfile
import json

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"
JSON_NAME = "dataset_comillas2.json"

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(JSON_NAME) as f:
        for i, line in enumerate(f):
            event = json.loads(line)

            if i < 5:
                print("\nEVENT", i)
                print("Top-level:", event.keys())
                print("_source keys:")
                print(list(event["_source"].keys()))

                # Search for anything mentioning attack / technique / MITRE
                matches = []
                def search_keys(obj, path=""):
                    if isinstance(obj, dict):
                        for k, v in obj.items():
                            p = f"{path}.{k}" if path else k
                            if any(x in k.lower() for x in
                                   ["attack", "technique", "tactic", "label",
                                    "malicious", "threat", "scenario"]):
                                matches.append((p, v))
                            search_keys(v, p)
                    elif isinstance(obj, list):
                        for j, v in enumerate(obj):
                            search_keys(v, f"{path}[{j}]")

                search_keys(event)
                print("Potential labels:")
                for x in matches:
                    print(x)

            if i >= 4:
                break


EVENT 0
Top-level: dict_keys(['_index', '_type', '_id', '_score', '_source'])
_source keys:
['event_original_time', 'etl_processed_time', 'user_name', 'PossibleCause', 'type', 'process_id', '@timestamp', 'host_name', 'level', 'etl_host_agent_type', 'etl_pipeline', 'task', 'User', 'etl_host_agent_ephemeral_uid', 'thread_id', 'user_domain', 'ClientMachine', '@version', 'Operation', 'z_elastic_ecs', 'xml_name', 'opcode', 'event_id', 'Component', 'ResultCode', 'log_name', 'ClientProcessId', 'record_number', 'event_original_message', 'provider_guid', 'beat_name', 'meta_user_name_is_machine', 'etl_host_agent_uid', 'source_name', 'etl_version', 'Id', 'beat_version']
Potential labels:

EVENT 1
Top-level: dict_keys(['_index', '_type', '_id', '_score', '_source'])
_source keys:
['event_original_time', 'etl_processed_time', 'user_name', 'PossibleCause', 'type', 'process_id', '@timestamp', 'host_name', 'level', 'etl_host_agent_type', 'etl_pipeline', 'task', 'User', 'etl_host_agent_ephemeral_uid',

In [ ]:
import zipfile
import json

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"
JSON_NAME = "dataset_comillas2.json"

keywords = [
    "mitre",
    "attack",
    "technique",
    "tactic",
    "T1059",
    "T1110",
    "T1003",
    "malicious",
]

found = 0

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(JSON_NAME) as f:
        for i, line in enumerate(f):
            text = line.decode("utf-8", errors="ignore").lower()

            matches = [k for k in keywords if k.lower() in text]

            if matches:
                event = json.loads(line)

                print(f"\nEVENT #{i}")
                print("Matches:", matches)
                print(json.dumps(event, indent=2)[:5000])

                found += 1

                if found >= 10:
                    break

print(f"\nFound {found} matching events.")


EVENT #395
Matches: ['mitre']
{
  "_index": "logs-endpoint-winevent-additional-2022.11.16",
  "_type": "_doc",
  "_id": "3033dad108c838d9dae35443845f9c0650ad7e94",
  "_score": 1,
  "_source": {
    "event_original_time": "2022-11-16T08:23:47.200Z",
    "etl_processed_time": "2022-11-16T08:28:24.761Z",
    "Message": "SLS Configuration:  {\n     \"AADAuthority\" : \"https://login.windows.net/common\",\n     \"AADProviderClientId\" : \"268761a2-03f3-40df-8a8b-c3db24145b6b\",\n     \"AADProviderScope\" : \"www.microsoft.com::mbi_ssl\",\n     \"AADResource\" : \"https://onestore.microsoft.com\",\n     \"AlternateId\" : \"https://storeedgefd.dsx.mp.microsoft.com/channels/products/{productId}?idType={idType}&appversion={appVersion}&market={marketCode}&locale={localeCode}&deviceType={deviceType}&deviceFamily={deviceFamily}&catalogLocales={catalogLocaleCodes}&musicMarket={musicMarketCode}&hardware={hardware}&packageHardware={packageHardware}&deviceFamilyVersion={deviceFamilyVersion}&architect

In [ ]:
import zipfile
import json

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

def find_mitre(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k

            if any(x in str(k).lower() for x in
                   ["mitre", "attack", "technique", "tactic", "label"]):
                print("KEY:", new_path)
                print("VALUE:", repr(v)[:1000])

            find_mitre(v, new_path)

    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            find_mitre(v, f"{path}[{i}]")

with zipfile.ZipFile(ZIP_PATH) as z:
    json_name = [x for x in z.namelist()
                 if x.lower().endswith(".json")][0]

    with z.open(json_name) as f:
        # Assuming JSONL for now
        for i, line in enumerate(f):
            if i >= 10000:
                break

            event = json.loads(line)
            find_mitre(event)

In [ ]:
import zipfile
import json
import re

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

patterns = re.compile(
    r"(T\d{4}(?:\.\d{3})?|MITRE|ATT&CK|technique|tactic)",
    re.IGNORECASE
)

with zipfile.ZipFile(ZIP_PATH) as z:
    print(z.namelist())

    json_name = [x for x in z.namelist() if x.lower().endswith(".json")][0]

    with z.open(json_name) as f:
        for i, line in enumerate(f):
            if i >= 10000:
                break

            event = json.loads(line)

            def search_values(obj, path=""):
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        search_values(v, f"{path}.{k}")
                elif isinstance(obj, list):
                    for j, v in enumerate(obj):
                        search_values(v, f"{path}[{j}]")
                elif isinstance(obj, str) and patterns.search(obj):
                    print("\nEVENT:", i)
                    print("PATH:", path)
                    print("MATCH:", patterns.findall(obj)[:20])
                    print("VALUE:", obj[:1000])

            search_values(event)

['dataset_comillas2.json']

EVENT: 25
PATH: ._source.DeviceInstanceId
MATCH: ['T3031']
VALUE: USB\VID_0781&PID_558C\MSFT30313931323739343031373435

EVENT: 25
PATH: ._source.event_original_message
MATCH: ['T3031']
VALUE: Device USB\VID_0781&PID_558C\MSFT30313931323739343031373435 was configured.

Driver Name: uaspstor.inf
Class Guid: {4D36E97B-E325-11CE-BFC1-08002BE10318}
Driver Date: 06/21/2006
Driver Version: 10.0.10240.16384
Driver Provider: Microsoft
Driver Section: UASPort_Install_Control
Driver Rank: 0xFF2000
Matching Device Id: USB\Class_08&SubClass_06&Prot_62
Outranked Drivers: 
Device Updated: false

EVENT: 26
PATH: ._source.DeviceInstanceId
MATCH: ['T3031']
VALUE: USB\VID_0781&PID_558C\MSFT30313931323739343031373435

EVENT: 26
PATH: ._source.event_original_message
MATCH: ['T3031']
VALUE: Device USB\VID_0781&PID_558C\MSFT30313931323739343031373435 was started.

Driver Name: uaspstor.inf
Class Guid: {4D36E97B-E325-11CE-BFC1-08002BE10318}
Service: UASPStor
Lower Filters: 
Upper F

In [ ]:
import zipfile
import json
from collections import Counter

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

key_counts = Counter()

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("dataset_comillas2.json") as f:
        for i, line in enumerate(f):
            if i >= 1000:
                break

            event = json.loads(line)

            if "_source" in event:
                key_counts.update(event["_source"].keys())

print("Most common fields:")
for key, count in key_counts.most_common():
    print(f"{key}: {count}")

Most common fields:
event_original_time: 1000
etl_processed_time: 1000
type: 1000
process_id: 1000
@timestamp: 1000
host_name: 1000
level: 1000
etl_host_agent_type: 1000
etl_pipeline: 1000
task: 1000
etl_host_agent_ephemeral_uid: 1000
thread_id: 1000
@version: 1000
z_elastic_ecs: 1000
event_id: 1000
log_name: 1000
record_number: 1000
event_original_message: 1000
provider_guid: 1000
beat_name: 1000
etl_host_agent_uid: 1000
source_name: 1000
etl_version: 1000
beat_version: 1000
opcode: 999
keywords: 735
Message: 466
Line Number: 466
Source: 466
Function: 466
activity_id: 230
Error Code: 207
TaskName: 197
version: 142
State Machine Name: 137
Thread ID: 137
State Machine: 137
Current State: 94
InstanceId: 85
Event Name: 84
TaskInstanceId: 66
UserContext: 65
EnginePID: 64
ActionName: 58
Id: 50
bytesTransferred: 49
bytesTransferredFromPeer: 49
ResultCode: 47
name: 36
url: 36
transferId: 36
fileLength: 36
bytesTotal: 36
fileTime: 36
User: 33
ProcessID: 31
xml_name: 28
serviceGuid: 28
Priority

In [ ]:
import zipfile
import json
from collections import Counter

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

counters = {
    "log_name": Counter(),
    "source_name": Counter(),
    "event_id": Counter(),
    "type": Counter(),
    "task": Counter(),
}

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("dataset_comillas2.json") as f:
        for i, line in enumerate(f):
            if i >= 100_000:
                break

            e = json.loads(line)["_source"]

            for field, counter in counters.items():
                value = e.get(field)
                if value is not None:
                    counter[str(value)] += 1

for field, counter in counters.items():
    print(f"\n===== {field} =====")
    for value, count in counter.most_common(30):
        print(f"{count:7}  {value}")


===== log_name =====
  43232  Microsoft-Windows-Store/Operational
  13273  Microsoft-Windows-TaskScheduler/Operational
   6541  Microsoft-Windows-RemoteDesktopServices-RdpCoreTS/Operational
   6051  Microsoft-Windows-Shell-Core/Operational
   5771  Microsoft-Windows-Windows Firewall With Advanced Security/Firewall
   4405  Microsoft-Windows-GroupPolicy/Operational
   3013  Microsoft-Windows-Kernel-PnP/Configuration
   2498  Microsoft-Windows-WMI-Activity/Operational
   2228  Microsoft-Windows-Bits-Client/Operational
   1546  Microsoft-Windows-WindowsUpdateClient/Operational
   1455  Microsoft-Windows-LiveId/Operational
   1268  Microsoft-Client-Licensing-Platform/Admin
    930  Microsoft-Windows-Winlogon/Operational
    835  Microsoft-Windows-SmbClient/Connectivity
    822  Microsoft-Windows-Shell-Core/AppDefaults
    782  Microsoft-Windows-TerminalServices-LocalSessionManager/Operational
    522  Microsoft-Windows-Storage-Storport/Health
    479  Microsoft-Windows-SMBServer/Operation

In [ ]:
import zipfile

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("dataset_comillas2.json") as f:
        first = f.readline()
        print("FIRST LINE:")
        print(first[:3000].decode("utf-8", errors="replace"))

        # Jump near the end of the compressed stream
        f.seek(0, 2)
        print("\nEND POSITION:", f.tell())


FIRST LINE:
{"_index":"logs-endpoint-winevent-additional-2022.11.16","_type":"_doc","_id":"8c0acd213cc3524fc717afbc33644d9c2eb6626c","_score":1,"_source":{"event_original_time":"2022-11-16T08:25:38.047Z","etl_processed_time":"2022-11-16T08:28:16.386Z","user_name":"system","PossibleCause":"Unknown","type":"wineventlog","process_id":888,"@timestamp":"2022-11-16T08:25:38.047Z","host_name":"desktop-4pvps6e.phoenix.local","level":"error","etl_host_agent_type":"winlogbeat","etl_pipeline":["all-filter-0098","all-add_processed_timestamp","fingerprint-winlogbeats7","winlogbeat_7_and_above-field_nest_cleanup","winlogbeat_7_and_above-field_cleanups","1500","1522","wmi-user_field_renames","wmi-all-extract_domain_and_user_name","general_rename-various_global_options","provider_guid-cleanup","winevent-hostname-cleanup","winevent-user_name-is-machine-account","final-cleanup-message_field"],"task":"None","User":"nt authority\\system","etl_host_agent_ephemeral_uid":"f7b6d19e-5167-4483-b795-09a22efb4701

In [ ]:
import zipfile, json, re

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"
pattern = re.compile(r'\bT\d{4}(?:\.\d{3})?\b')

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("dataset_comillas2.json") as f:
        for i, line in enumerate(f):
            event = json.loads(line)

            matches = []

            def scan(x, path=""):
                if isinstance(x, dict):
                    for k, v in x.items():
                        scan(v, path + "." + k)
                elif isinstance(x, list):
                    for j, v in enumerate(x):
                        scan(v, path + f"[{j}]")
                elif isinstance(x, str):
                    for m in pattern.findall(x):
                        matches.append((m, path))

            scan(event)

            # Ignore IDs embedded in USB/device identifiers
            real = [(m, p) for m, p in matches
                    if "DeviceInstanceId" not in p]

            if real:
                print("\nEVENT", i)
                print(real[:20])
                print(json.dumps(event["_source"], indent=2)[:5000])
                break


EVENT 431058
[('T1553.004', '._source.RuleName'), ('T1553.004', '._source.event_original_message'), ('T1553.004', '._source.rule_technique_id')]
{
  "event_original_time": "2022-11-16T19:59:51.007Z",
  "etl_processed_time": "2022-11-16T20:00:30.683Z",
  "user_name": "louis",
  "process_guid": "4FD6B357-412E-6375-CA00-000000002900",
  "type": "wineventlog",
  "version": 2,
  "rule_technique_name": "Install Root Certificate",
  "process_id": "1120",
  "@timestamp": "2022-11-16T19:59:51.007Z",
  "host_name": "desktop-4pvps6e.phoenix.local",
  "process_path": "c:\\users\\louis\\appdata\\local\\microsoft\\onedrive\\onedrive.exe",
  "level": "information",
  "etl_host_agent_type": "winlogbeat",
  "etl_pipeline": [
    "all-filter-0098",
    "all-add_processed_timestamp",
    "fingerprint-winlogbeats7",
    "winlogbeat_7_and_above-field_nest_cleanup",
    "winlogbeat_7_and_above-field_cleanups",
    "1500",
    "1522",
    "winevent-sysmon-all-1531",
    "sysmon-all-extract_domain_and_user_n

In [ ]:
import zipfile
import json
import gzip
import time

ZIP_PATH = "/content/Comiset23_Lab_Environment_Dataset.zip"
OUTPUT_PATH = "/content/comiset_lab_subset.jsonl.gz"

benign_seen = 0
benign_kept = 0
malicious_kept = 0
total_seen = 0

start = time.time()

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("dataset_comillas2.json", "r") as src, \
         gzip.open(OUTPUT_PATH, "wt", encoding="utf-8") as dst:

        for line in src:
            total_seen += 1

            event = json.loads(line)
            source = event.get("_source", {})

            # ATT&CK-labeled = malicious
            technique = source.get("rule_technique_id")

            if technique:
                # Keep ALL malicious events
                dst.write(json.dumps(event, separators=(",", ":")) + "\n")
                malicious_kept += 1

            else:
                # Keep exactly ~50% of benign events
                if benign_seen % 2 == 0:
                    dst.write(json.dumps(event, separators=(",", ":")) + "\n")
                    benign_kept += 1

                benign_seen += 1

            if total_seen % 1_000_000 == 0:
                elapsed = time.time() - start
                print(
                    f"{total_seen:,} processed | "
                    f"{benign_kept:,} benign kept | "
                    f"{malicious_kept:,} malicious kept | "
                    f"{elapsed/60:.1f} min"
                )

print("\nDONE")
print(f"Total processed:     {total_seen:,}")
print(f"Benign kept:         {benign_kept:,}")
print(f"Malicious kept:      {malicious_kept:,}")
print(f"Total output events: {benign_kept + malicious_kept:,}")
print(f"Output:              {OUTPUT_PATH}")

1,000,000 processed | 498,420 benign kept | 3,161 malicious kept | 1.7 min
2,000,000 processed | 994,400 benign kept | 11,201 malicious kept | 3.4 min
3,000,000 processed | 1,481,439 benign kept | 37,123 malicious kept | 5.3 min
4,000,000 processed | 1,933,776 benign kept | 132,449 malicious kept | 7.4 min
5,000,000 processed | 2,403,718 benign kept | 192,565 malicious kept | 9.2 min
6,000,000 processed | 2,870,279 benign kept | 259,442 malicious kept | 11.0 min
7,000,000 processed | 3,361,948 benign kept | 276,105 malicious kept | 12.8 min
8,000,000 processed | 3,779,826 benign kept | 440,349 malicious kept | 14.9 min
9,000,000 processed | 4,246,065 benign kept | 507,871 malicious kept | 16.9 min
10,000,000 processed | 4,739,282 benign kept | 521,436 malicious kept | 18.6 min
11,000,000 processed | 5,234,089 benign kept | 531,822 malicious kept | 20.5 min
12,000,000 processed | 5,716,221 benign kept | 567,558 malicious kept | 22.2 min
13,000,000 processed | 6,188,806 benign kept | 622

In [ ]:
import os

size_gb = os.path.getsize("/content/comiset_lab_subset.jsonl.gz") / (1024**3)
print(f"{size_gb:.2f} GB")

2.60 GB


In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_dc4d5b3156c7d8449d6ea40c1a53b0e9"

In [ ]:

!pip install -q kaggle

!kaggle datasets init -p /content


Data package template written to: /content/dataset-metadata.json


In [ ]:
import json

path = "/content/dataset-metadata.json"

metadata = {
    "title": "COMISET Lab 50% Benign Subset",
    "id": "saifthesaif/comiset-lab-50-benign-subset",
    "licenses": [{"name": "CC0-1.0"}],
    "resources": [
        {
            "path": "comiset_lab_subset.jsonl.gz"
        }
    ]
}

with open(path, "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
!cp /content/comiset_lab_subset.jsonl.gz /content/
!kaggle datasets create -p /content

cp: '/content/comiset_lab_subset.jsonl.gz' and '/content/comiset_lab_subset.jsonl.gz' are the same file
Skipping folder: .config; use '--dir-mode' to upload folders
Starting upload for file comiset_lab_subset.jsonl.gz
100% 2.60G/2.60G [00:24<00:00, 116MB/s]
Upload successful: comiset_lab_subset.jsonl.gz (3GB)
Skipping folder: sample_data; use '--dir-mode' to upload folders
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/saifthesaif/comiset-lab-50-benign-subset


In [ ]:
!kaggle datasets list -s "COMISET Lab 50% Benign Subset"

No datasets found


In [ ]:
!rm /content/Comiset23_Lab_Environment_Dataset.zip

==========================================================================================================================================================================================================================================================================================================================================================


In [ ]:
import gzip
import json
from collections import Counter

PATH = "/content/comiset_lab_subset.jsonl.gz"
N = 1000_000

key_counts = Counter()

with gzip.open(PATH, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break

        event = json.loads(line)
        key_counts.update(event["_source"].keys())

print(f"{len(key_counts)} unique keys\n")
j = 0
for key, count in key_counts.most_common():
    j += 1
    percentage = count / N * 100
    print(f"{j} {percentage:6.2f}%  {count:5,}  {key}")

1097 unique keys

1 100.00%  1,000,000  event_original_time
2 100.00%  1,000,000  type
3 100.00%  1,000,000  @timestamp
4 100.00%  1,000,000  host_name
5 100.00%  1,000,000  level
6 100.00%  1,000,000  etl_host_agent_type
7 100.00%  1,000,000  etl_pipeline
8 100.00%  1,000,000  etl_host_agent_ephemeral_uid
9 100.00%  1,000,000  @version
10 100.00%  1,000,000  z_elastic_ecs
11 100.00%  1,000,000  event_id
12 100.00%  1,000,000  log_name
13 100.00%  1,000,000  record_number
14 100.00%  1,000,000  beat_name
15 100.00%  1,000,000  etl_host_agent_uid
16 100.00%  1,000,000  source_name
17 100.00%  1,000,000  beat_version
18 100.00%  999,994  task
19  99.98%  999,828  etl_processed_time
20  99.98%  999,828  event_original_message
21  99.98%  999,828  etl_version
22  99.89%  998,930  opcode
23  99.40%  994,044  provider_guid
24  98.96%  989,597  process_id
25  98.96%  989,597  thread_id
26  80.91%  809,148  process_path
27  80.91%  809,148  process_name
28  80.47%  804,747  version
29  79.27% 

In [ ]:
import gzip
import json
from collections import Counter, defaultdict

PATH = "/content/comiset_lab_subset.jsonl.gz"

N = 1000_000  # increase later if needed

benign_counts = Counter()
malicious_counts = Counter()

benign_total = 0
malicious_total = 0

with gzip.open(PATH, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break

        event = json.loads(line)
        source = event["_source"]

        # Dataset label
        malicious = bool(source.get("rule_technique_id"))

        target = malicious_counts if malicious else benign_counts

        for key in source:
            target[key] += 1

        if malicious:
            malicious_total += 1
        else:
            benign_total += 1

print(f"Events analyzed: {benign_total + malicious_total:,}")
print(f"Benign:          {benign_total:,}")
print(f"Malicious:       {malicious_total:,}")

# Compare field presence
all_keys = set(benign_counts) | set(malicious_counts)

results = []

for key in all_keys:
    b = benign_counts[key]
    m = malicious_counts[key]

    b_pct = 100 * b / benign_total if benign_total else 0
    m_pct = 100 * m / malicious_total if malicious_total else 0

    results.append((key, b_pct, m_pct, m_pct - b_pct))

# Most characteristic of malicious events
results.sort(key=lambda x: x[3], reverse=True)

print("\nFields much more common in MALICIOUS events:")
print(f"{'Field':45} {'Benign %':>10} {'Malicious %':>12} {'Difference':>12}")
print("-" * 82)

for key, b_pct, m_pct, diff in results[:100]:
    print(f"{key[:45]:45} {b_pct:9.2f}% {m_pct:11.2f}% {diff:+11.2f}%")

Events analyzed: 1,000,000
Benign:          988,932
Malicious:       11,068

Fields much more common in MALICIOUS events:
Field                                           Benign %  Malicious %   Difference
----------------------------------------------------------------------------------
rule_technique_id                                  0.00%      100.00%     +100.00%
rule_technique_name                                0.00%       99.93%      +99.92%
user_account                                      60.85%       95.37%      +34.51%
user_domain                                       63.15%       95.37%      +32.21%
user_name                                         63.15%       95.37%      +32.21%
meta_user_name_is_machine                         63.15%       95.37%      +32.21%
Details                                            0.45%       32.08%      +31.63%
file_creation_time                                 0.08%       25.47%      +25.39%
TargetFilename                                  

=======================================================================================================================================================================================================


In [ ]:
# Example download command
!kaggle datasets download -d saifthesaif/comiset-lab-50-benign-subset
!unzip -o comiset-lab-50-benign-subset.zip -d dataset/
!rm comiset-lab-50-benign-subset.zip


Dataset URL: https://www.kaggle.com/datasets/saifthesaif/comiset-lab-50-benign-subset
License(s): CC0-1.0
100% 2.84G/2.84G [00:34<00:00, 89.8MB/s]

Archive:  comiset-lab-50-benign-subset.zip
  inflating: dataset/comiset_lab_subset.jsonl  


In [ ]:
import json
from collections import Counter

PATH = "/content/dataset/comiset_lab_subset.jsonl"

N = 1_000_000  # increase later if needed

benign_counts = Counter()
malicious_counts = Counter()

benign_total = 0
malicious_total = 0

with open(PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break

        event = json.loads(line)
        source = event["_source"]

        # Dataset label
        malicious = bool(source.get("rule_technique_id"))

        target = malicious_counts if malicious else benign_counts

        for key in source:
            target[key] += 1

        if malicious:
            malicious_total += 1
        else:
            benign_total += 1

print(f"Events analyzed: {benign_total + malicious_total:,}")
print(f"Benign:          {benign_total:,}")
print(f"Malicious:       {malicious_total:,}")

# Compare field presence
all_keys = set(benign_counts) | set(malicious_counts)

results = []

for key in all_keys:
    b = benign_counts[key]
    m = malicious_counts[key]

    b_pct = 100 * b / benign_total if benign_total else 0
    m_pct = 100 * m / malicious_total if malicious_total else 0

    results.append((key, b_pct, m_pct, m_pct - b_pct))

# Most characteristic of malicious events
results.sort(key=lambda x: x[3], reverse=True)

print("\nFields much more common in MALICIOUS events:")
print(f"{'Field':45} {'Benign %':>10} {'Malicious %':>12} {'Difference':>12}")
print("-" * 82)

for key, b_pct, m_pct, diff in results[:100]:
    print(f"{key[:45]:45} {b_pct:9.2f}% {m_pct:11.2f}% {diff:+11.2f}%")

KeyboardInterrupt: 

In [12]:
import json

PATH = "/content/dataset/comiset_lab_subset.jsonl"

# For each field:
# first_values[key] = first value encountered
# non_constant[key] = True if a different value is later encountered

first_values = {}
non_constant = set()

with open(PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        event = json.loads(line)
        source = event["_source"]

        for key, value in source.items():
            # First occurrence of this field
            if key not in first_values:
                first_values[key] = value

            # If we already know it varies, no need to check further
            elif key not in non_constant and value != first_values[key]:
                non_constant.add(key)

        if (i + 1) % 1_000_000 == 0:
            print(f"Processed {i + 1:,} events")

constant_cols = [
    key for key in first_values
    if key not in non_constant
]

print(f"\nTotal fields:     {len(first_values)}")
print(f"Constant fields:  {len(constant_cols)}")
print(f"Remaining fields: {len(first_values) - len(constant_cols)}")

print("\nConstant fields:")
for key in constant_cols:
    print(f"{key}: {first_values[key]!r}")

Processed 1,000,000 events
Processed 2,000,000 events
Processed 3,000,000 events
Processed 4,000,000 events
Processed 5,000,000 events
Processed 6,000,000 events
Processed 7,000,000 events
Processed 8,000,000 events
Processed 9,000,000 events
Processed 10,000,000 events
Processed 11,000,000 events
Processed 12,000,000 events
Processed 13,000,000 events
Processed 14,000,000 events
Processed 15,000,000 events
Processed 16,000,000 events
Processed 17,000,000 events
Processed 18,000,000 events
Processed 19,000,000 events
Processed 20,000,000 events
Processed 21,000,000 events
Processed 22,000,000 events
Processed 23,000,000 events
Processed 24,000,000 events
Processed 25,000,000 events

Total fields:     1249
Constant fields:  459
Remaining fields: 790

Constant fields:
type: 'wineventlog'
etl_host_agent_type: 'winlogbeat'
ClientMachine: 'DESKTOP-4PVPS6E'
@version: '1'
beat_name: 'DESKTOP-4PVPS6E'
etl_host_agent_uid: 'f99bee41-708a-4aa2-ad36-b5a565e36b68'
etl_version: '2020.04.19.01'
beat_

In [18]:
import json
from collections import defaultdict

PATH = "/content/dataset/comiset_lab_subset.jsonl"
N = 2_000_000

# Track values for non-constant fields
values = defaultdict(set)
counts = defaultdict(int)

with open(PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N:
            break

        source = json.loads(line)["_source"]

        for key, value in source.items():
            if key in constant_cols:
                continue

            counts[key] += 1

            # Don't let pathological fields consume unlimited RAM
            if len(values[key]) <= 100_000:
                values[key].add(str(value))

        if (i + 1) % 100_000 == 0:
            print(f"Processed {i + 1:,}")

# Find extremely high-cardinality fields
candidates = []

for key in counts:
    unique = len(values[key])
    total = counts[key]

    # Only meaningful if we didn't hit the safety cap
    if unique <= 100_000:
        ratio = unique / total

        if ratio >= 0.99:
            candidates.append((key, total, unique, ratio))

candidates.sort(key=lambda x: x[3], reverse=True)

print("\nPotential pure-ID fields:")
print(f"{'Field':45} {'Count':>10} {'Unique':>10} {'Unique %':>10}")
print("-" * 80)

for key, total, unique, ratio in candidates:
    print(f"{key[:45]:45} {total:10,} {unique:10,} {ratio*100:9.2f}%")


Processed 100,000
Processed 200,000
Processed 300,000
Processed 400,000
Processed 500,000
Processed 600,000
Processed 700,000
Processed 800,000
Processed 900,000
Processed 1,000,000
Processed 1,100,000
Processed 1,200,000
Processed 1,300,000
Processed 1,400,000
Processed 1,500,000
Processed 1,600,000
Processed 1,700,000
Processed 1,800,000
Processed 1,900,000
Processed 2,000,000

Potential pure-ID fields:
Field                                              Count     Unique   Unique %
--------------------------------------------------------------------------------
PolicyActivityId                                     240        240    100.00%
ErrorSourceTable                                      70         70    100.00%
OperationParameter1                                  279        279    100.00%
GpsvcTimeElapsedInMilliseconds                        22         22    100.00%
PolicyDownloadTimeElapsedInMilliseconds               60         60    100.00%
NewDUID                             

In [19]:
candidates = [
    x for x in candidates
    if x[1] >= 20
]

In [20]:
print(f"{'Field':45} {'Count':>10} {'Unique':>10} {'Unique %':>10}")
print("-" * 80)

for key, total, unique, ratio in candidates:
    print(f"{key[:45]:45} {total:10,} {unique:10,} {ratio*100:9.2f}%")

Field                                              Count     Unique   Unique %
--------------------------------------------------------------------------------
PolicyActivityId                                     240        240    100.00%
ErrorSourceTable                                      70         70    100.00%
OperationParameter1                                  279        279    100.00%
GpsvcTimeElapsedInMilliseconds                        22         22    100.00%
PolicyDownloadTimeElapsedInMilliseconds               60         60    100.00%
Srb                                                   53         53    100.00%
BucketIoSuccess2                                      94         94    100.00%
TotalIoCount                                          94         94    100.00%
BucketIoLatency3_100ns                                94         94    100.00%
MaxReadWriteLatency_100ns                             94         94    100.00%
BucketIoLatency1_100ns                            

In [21]:
import gzip
import json
import os

INPUT = "/content/dataset/comiset_lab_subset.jsonl"
OUTPUT = "/content/dataset/comiset_lab_cleaned.jsonl.gz"

remove_cols = set(constant_cols)
remove_cols.update([
    "record_number",
    # add confirmed useless IDs here
])

with open(INPUT, "r", encoding="utf-8") as fin, \
     gzip.open(OUTPUT, "wt", encoding="utf-8") as fout:

    for i, line in enumerate(fin):
        event = json.loads(line)

        for col in remove_cols:
            event["_source"].pop(col, None)

        fout.write(json.dumps(event, separators=(",", ":")) + "\n")

        if (i + 1) % 1_000_000 == 0:
            print(f"Processed {i + 1:,}")

print("Finished.")
print(f"Output size: {os.path.getsize(OUTPUT) / 1024**3:.2f} GB")

Processed 1,000,000
Processed 2,000,000
Processed 3,000,000
Processed 4,000,000
Processed 5,000,000
Processed 6,000,000
Processed 7,000,000
Processed 8,000,000
Processed 9,000,000
Processed 10,000,000
Processed 11,000,000
Processed 12,000,000
Processed 13,000,000
Processed 14,000,000
Processed 15,000,000
Processed 16,000,000
Processed 17,000,000
Processed 18,000,000
Processed 19,000,000
Processed 20,000,000
Processed 21,000,000
Processed 22,000,000
Processed 23,000,000
Processed 24,000,000
Processed 25,000,000
Finished.
Output size: 2.41 GB


In [22]:
from google.colab import files

files.download("/content/dataset/comiset_lab_cleaned.jsonl.gz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>